# Buscaminas MARIE - Notebook autosuficiente

Este notebook:

1. genera un tablero de Buscaminas 16×16,
2. genera una **máscara de forma** (`SHAPE_PRESET`) para cambiar la forma del tablero,
3. crea los archivos:
   - `marie/generados/buscaminas_generado_mod.mas`
   - `marie/generados/tablero_marie_generado.mem`
   - `marie/generados/shape_preset_generado.mem`
   - `outputs/vista_tablero_buscaminas.png`
4. deja todo listo para abrir el `.mas` en **Marie.js**.

Está pensado para funcionar tanto en **Google Colab** como en **local**.


In [ ]:

from __future__ import annotations

from pathlib import Path
from typing import List, Optional, Sequence, Tuple
import random
import os

try:
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
except Exception:
    !pip -q install matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap

# Crear carpetas de salida
Path("marie/generados").mkdir(parents=True, exist_ok=True)
Path("outputs").mkdir(parents=True, exist_ok=True)

print("Entorno preparado.")


## Parámetros del tablero y de la forma
Modifica esta celda para cambiar minas, semilla y forma.

In [ ]:

FILAS = 16
COLUMNAS = 16

# Puedes usar minas aleatorias con semilla...
MINAS = 20
SEMILLA = 42

# ...o definir minas fijas manualmente (pon una lista de tuplas o deja None)
MINAS_FIJAS = None
# Ejemplo:
# MINAS_FIJAS = [(0,0), (1,1), (5,7), (10,10)]

# Forma del tablero:
# "llena", "rombo", "cruz", "marco", "personalizada"
FORMA = "rombo"

# Parámetros adicionales de forma
GROSOR = 2

# Si FORMA = "personalizada", define aquí las celdas activas
ACTIVAS = None
# Ejemplo:
# ACTIVAS = [(7,7), (7,8), (8,7), (8,8)]


In [ ]:

Tablero = List[List[int]]
Mascara = List[List[int]]
Coordenada = Tuple[int, int]

def crear_tablero(filas: int = 16, columnas: int = 16) -> Tablero:
    return [[0 for _ in range(columnas)] for _ in range(filas)]

def validar_coordenada(fila: int, columna: int, filas: int, columnas: int) -> bool:
    return 0 <= fila < filas and 0 <= columna < columnas

def vecinos_de(fila: int, columna: int, filas: int, columnas: int) -> List[Coordenada]:
    resultado: List[Coordenada] = []
    for df in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if df == 0 and dc == 0:
                continue
            nf, nc = fila + df, columna + dc
            if validar_coordenada(nf, nc, filas, columnas):
                resultado.append((nf, nc))
    return resultado

def colocar_minas(
    tablero: Tablero,
    minas: int,
    semilla: Optional[int] = None,
    minas_fijas: Optional[Sequence[Coordenada]] = None,
) -> Tablero:
    filas = len(tablero)
    columnas = len(tablero[0]) if filas > 0 else 0

    if minas_fijas is not None:
        usadas = set()
        for fila, columna in minas_fijas:
            if not validar_coordenada(fila, columna, filas, columnas):
                raise ValueError(f"Coordenada de mina fuera de rango: {(fila, columna)}")
            if (fila, columna) in usadas:
                continue
            tablero[fila][columna] = -1
            usadas.add((fila, columna))
        return tablero

    if minas < 0 or minas > filas * columnas:
        raise ValueError("La cantidad de minas es inválida para el tamaño del tablero.")

    rng = random.Random(semilla)
    colocadas = 0
    while colocadas < minas:
        fila = rng.randint(0, filas - 1)
        columna = rng.randint(0, columnas - 1)
        if tablero[fila][columna] != -1:
            tablero[fila][columna] = -1
            colocadas += 1

    return tablero

def calcular_vecinos(tablero: Tablero) -> Tablero:
    filas = len(tablero)
    columnas = len(tablero[0]) if filas > 0 else 0

    for fila in range(filas):
        for columna in range(columnas):
            if tablero[fila][columna] == -1:
                continue

            conteo = 0
            for nf, nc in vecinos_de(fila, columna, filas, columnas):
                if tablero[nf][nc] == -1:
                    conteo += 1

            tablero[fila][columna] = conteo

    return tablero

def generar_tablero(
    filas: int = 16,
    columnas: int = 16,
    minas: int = 20,
    semilla: Optional[int] = None,
    minas_fijas: Optional[Sequence[Coordenada]] = None,
) -> Tablero:
    tablero = crear_tablero(filas, columnas)
    tablero = colocar_minas(tablero, minas=minas, semilla=semilla, minas_fijas=minas_fijas)
    tablero = calcular_vecinos(tablero)
    return tablero

# ==========================================
# Máscaras de forma
# ==========================================
def crear_mascara_llena(filas: int = 16, columnas: int = 16) -> Mascara:
    return [[1 for _ in range(columnas)] for _ in range(filas)]

def crear_mascara_rombo(filas: int = 16, columnas: int = 16) -> Mascara:
    centro_f = filas // 2
    centro_c = columnas // 2
    radio = min(filas, columnas) // 2
    mascara = [[0 for _ in range(columnas)] for _ in range(filas)]
    for i in range(filas):
        for j in range(columnas):
            if abs(i - centro_f) + abs(j - centro_c) <= radio:
                mascara[i][j] = 1
    return mascara

def crear_mascara_cruz(filas: int = 16, columnas: int = 16, grosor: int = 2) -> Mascara:
    centro_f = filas // 2
    centro_c = columnas // 2
    mascara = [[0 for _ in range(columnas)] for _ in range(filas)]
    for i in range(filas):
        for j in range(columnas):
            if abs(i - centro_f) <= grosor or abs(j - centro_c) <= grosor:
                mascara[i][j] = 1
    return mascara

def crear_mascara_marco(filas: int = 16, columnas: int = 16, grosor: int = 2) -> Mascara:
    mascara = [[0 for _ in range(columnas)] for _ in range(filas)]
    for i in range(filas):
        for j in range(columnas):
            if i < grosor or i >= filas - grosor or j < grosor or j >= columnas - grosor:
                mascara[i][j] = 1
    return mascara

def crear_mascara_personalizada(
    activas: Sequence[Coordenada],
    filas: int = 16,
    columnas: int = 16,
) -> Mascara:
    mascara = [[0 for _ in range(columnas)] for _ in range(filas)]
    for fila, columna in activas:
        if not validar_coordenada(fila, columna, filas, columnas):
            raise ValueError(f"Coordenada fuera de rango: {(fila, columna)}")
        mascara[fila][columna] = 1
    return mascara

def generar_mascara(
    forma: str = "llena",
    filas: int = 16,
    columnas: int = 16,
    grosor: int = 2,
    activas: Optional[Sequence[Coordenada]] = None,
) -> Mascara:
    forma = forma.lower().strip()

    if forma == "llena":
        return crear_mascara_llena(filas, columnas)
    if forma == "rombo":
        return crear_mascara_rombo(filas, columnas)
    if forma == "cruz":
        return crear_mascara_cruz(filas, columnas, grosor=grosor)
    if forma == "marco":
        return crear_mascara_marco(filas, columnas, grosor=grosor)
    if forma == "personalizada":
        if activas is None:
            raise ValueError("Para forma 'personalizada' debes enviar 'activas'.")
        return crear_mascara_personalizada(activas, filas, columnas)

    raise ValueError(f"Forma no soportada: {forma}")

def tablero_a_texto(tablero: Tablero, mascara: Optional[Mascara] = None) -> str:
    lineas = []
    filas = len(tablero)
    columnas = len(tablero[0]) if filas > 0 else 0

    for i in range(filas):
        partes = []
        for j in range(columnas):
            if mascara is not None and mascara[i][j] == 0:
                partes.append(".")
            else:
                valor = tablero[i][j]
                partes.append("*" if valor == -1 else str(valor))
        lineas.append(" ".join(partes))
    return "\n".join(lineas)

def guardar_visualizacion(
    tablero: Tablero,
    ruta_salida: str | Path,
    mascara: Optional[Mascara] = None,
    titulo: str = "Vista previa del tablero Buscaminas",
) -> Path:
    ruta = Path(ruta_salida)

    matriz = []
    for i in range(len(tablero)):
        fila_out = []
        for j in range(len(tablero[0])):
            if mascara is not None and mascara[i][j] == 0:
                fila_out.append(10)
            else:
                valor = tablero[i][j]
                fila_out.append(9 if valor == -1 else valor)
        matriz.append(fila_out)

    cmap = ListedColormap([
        "#d9d9d9",  # 0
        "#0000ff",  # 1
        "#008000",  # 2
        "#ff0000",  # 3
        "#00008b",  # 4
        "#8b0000",  # 5
        "#00ced1",  # 6
        "#8a2be2",  # 7
        "#ffffff",  # 8
        "#000000",  # mina
        "#333333",  # fuera de forma
    ])

    plt.figure(figsize=(8, 8))
    plt.imshow(matriz, cmap=cmap, vmin=0, vmax=10)

    filas = len(matriz)
    columnas = len(matriz[0]) if filas > 0 else 0

    for i in range(filas):
        for j in range(columnas):
            if mascara is not None and mascara[i][j] == 0:
                texto = ""
                color_texto = "white"
            else:
                valor = tablero[i][j]
                texto = "*" if valor == -1 else str(valor)
                color_texto = "white" if valor == -1 else "black"

            plt.text(j, i, texto, ha="center", va="center", fontsize=8, color=color_texto)

    plt.title(titulo)
    plt.xticks(range(columnas))
    plt.yticks(range(filas))
    plt.grid(True, which="both", color="gray", linewidth=0.5)
    plt.tight_layout()
    plt.savefig(ruta, dpi=200)
    plt.close()

    return ruta

def a_lineal(matriz: List[List[int]]) -> List[int]:
    return [valor for fila in matriz for valor in fila]

def bloque_dec_desde_matriz(matriz: List[List[int]]) -> str:
    return "\n".join(f"    DEC {valor}" for valor in a_lineal(matriz))

def exportar_mem(tablero: Tablero, ruta_salida: str | Path) -> Path:
    ruta = Path(ruta_salida)
    contenido = "TABLERO_REAL,\n" + bloque_dec_desde_matriz(tablero) + "\n"
    ruta.write_text(contenido, encoding="utf-8")
    return ruta

def exportar_shape_mem(mascara: Mascara, ruta_salida: str | Path) -> Path:
    ruta = Path(ruta_salida)
    contenido = "SHAPE_PRESET,\n" + bloque_dec_desde_matriz(mascara) + "\n"
    ruta.write_text(contenido, encoding="utf-8")
    return ruta


## Plantilla `.mas` embebida
Esta plantilla ya incluye los marcadores `__TABLERO_REAL__` y `__SHAPE_PRESET__`.

In [ ]:
template_mas = r"""
/ ================================================================
/ BUSCAMINAS EN MARIE.JS
/ PLANTILLA MUY COMENTADA
/ ================================================================
/ Esta plantilla está pensada para que Python inserte:
/   1) TABLERO_REAL   -> valores -1, 0..8
/   2) SHAPE_PRESET   -> máscara 0/1 para definir la forma jugable
/
/ IDEA DE DISEÑO
/ - El tablero físico siempre es 16x16 = 256 posiciones.
/ - Pero no todas tienen que estar activas.
/ - SHAPE_PRESET define qué casillas existen:
/
/      1 = la celda pertenece a la forma del tablero
/      0 = la celda queda fuera del tablero visible/jugable
/
/ Esto te permite cambiar la "forma" del tablero sin tocar la lógica
/ principal del juego. Solo cambias la máscara generada desde Python.
/ ================================================================

ORG 100

/ =========================
/ CONSTANTES BÁSICAS
/ =========================
ZERO,           DEC 0
ONE,            DEC 1
TWO,            DEC 2
THREE,          DEC 3
FOUR,           DEC 4
FIVE,           DEC 5
SIX,            DEC 6
SEVEN,          DEC 7
EIGHT,          DEC 8
NINE,           DEC 9
TEN,            DEC 10
SIXTEEN,        DEC 16
NEGONE,         DEC -1
NEGTWO,         DEC -2
NEG255,         DEC -255
POS255,         DEC 255
DEC255,         DEC 255
DEC256,         DEC 256

/ =========================
/ COLORES DEL DISPLAY
/ =========================
/ Ajusta estos valores si tu Marie.js usa otra codificación.
COLOR_OCULTO,   HEX 444
COLOR_BANDERA,  HEX FF0
COLOR_MINA,     HEX 000
COLOR_0,        HEX CCC
COLOR_1,        HEX 00F
COLOR_2,        HEX 0A0
COLOR_3,        HEX F00
COLOR_4,        HEX 00A
COLOR_5,        HEX 800
COLOR_6,        HEX 0FF
COLOR_7,        HEX A0A
COLOR_8,        HEX FFF
COLOR_FORA,     HEX 222

/ ================================================================
/ MENSAJES (ASCII DECIMAL)
/ ================================================================
/ START
CH_S, DEC 83
CH_T, DEC 84
CH_A, DEC 65
CH_R, DEC 82
CH_SPACE, DEC 32
CH_G, DEC 71
CH_M, DEC 77
CH_E, DEC 69
CH_W, DEC 87
CH_I, DEC 73
CH_N, DEC 78
CH_L, DEC 76
CH_O, DEC 79
CH_F, DEC 70
CH_COLON, DEC 58
CH_STAR, DEC 42
CH_NEWLINE, DEC 10

/ ================================================================
/ DISPLAY 16x16
/ ================================================================
/ IMPORTANTE:
/
/ En muchas versiones de Marie.js, el display gráfico 16x16 comienza
/ en una dirección fija. Aquí asumimos:
/
/     DISP_START = 0F00
/
/ Si en tu simulador no funciona, cambia este valor.
DISP_START,      HEX 0F00

/ ================================================================
/ VARIABLES DE JUEGO
/ ================================================================
FILA,            DEC 0
COLUMNA,         DEC 0
ACCION,          DEC 0

GAME_STATE,      DEC 0
/ 0 = jugando
/ 1 = perdió
/ 2 = ganó

FLAG_COUNT,      DEC 0
REVEALED_COUNT,  DEC 0
SAFE_TARGET,     DEC 0

COUNT,           DEC 0
DIR,             DEC 0
VALOR,           DEC 0
TMP,             DEC 0
TMP2,            DEC 0
TMP3,            DEC 0

ROWTMP,          DEC 0
COLTMP,          DEC 0
INDEXTMP,        DEC 0

PTR_REAL,        DEC 0
PTR_VIS,         DEC 0
PTR_SHAPE,       DEC 0
PTR_DISP,        DEC 0

BASE_REAL,       DEC 0
BASE_VIS,        DEC 0
BASE_SHAPE,      DEC 0
BASE_DISP,       DEC 0

/ ================================================================
/ TABLERO VISIBLE
/ ================================================================
/ 0 = oculta
/ 1 = revelada
/ 2 = bandera
TABLERO_VISIBLE,
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0
    DEC 0

/ ================================================================
/ TABLERO REAL
/ Python reemplaza __TABLERO_REAL__
/ ================================================================
TABLERO_REAL,
__TABLERO_REAL__

/ ================================================================
/ SHAPE PRESET
/ Python reemplaza __SHAPE_PRESET__
/ ================================================================
SHAPE_PRESET,
__SHAPE_PRESET__

/ ================================================================
/ INICIO DEL PROGRAMA
/ ================================================================
START,          LOAD TABLERO_REAL
                STORE BASE_REAL
                LOAD TABLERO_VISIBLE
                STORE BASE_VIS
                LOAD SHAPE_PRESET
                STORE BASE_SHAPE
                LOAD DISP_START
                STORE BASE_DISP

                JNS PRINT_START
                JNS CALC_SAFE_TARGET
                JNS RENDER_BOARD

MAIN_LOOP,      LOAD GAME_STATE
                SKIPCOND 400
                JUMP CHECK_WIN_STATE
                JUMP READ_INPUTS

CHECK_WIN_STATE, LOAD GAME_STATE
                 SUBT TWO
                 SKIPCOND 400
                 JUMP END_GAME
                 JUMP READ_INPUTS

READ_INPUTS,    INPUT
                STORE FILA
                INPUT
                STORE COLUMNA
                INPUT
                STORE ACCION

                JNS CALC_INDEX
                JNS VALIDATE_SHAPE_CELL
                LOAD TMP
                SKIPCOND 400
                JUMP MAIN_LOOP

                LOAD ACCION
                SUBT ONE
                SKIPCOND 400
                JUMP CHECK_FLAG_ACTION
                JNS REVEAL_CELL
                JUMP AFTER_ACTION

CHECK_FLAG_ACTION, LOAD ACCION
                   SUBT TWO
                   SKIPCOND 400
                   JUMP AFTER_ACTION
                   JNS TOGGLE_FLAG

AFTER_ACTION,   JNS CHECK_WIN
                JNS RENDER_BOARD
                JUMP MAIN_LOOP

END_GAME,       LOAD GAME_STATE
                SUBT TWO
                SKIPCOND 400
                JUMP PRINT_LOSE_AND_STOP
                JNS PRINT_WIN
                HALT

PRINT_LOSE_AND_STOP, JNS PRINT_LOSE
                     HALT

/ ================================================================
/ CALC_INDEX
/ Convierte (fila, columna) a índice lineal: fila*16 + columna
/ Resultado en INDEXTMP
/ ================================================================
CALC_INDEX,     HEX 0
                LOAD FILA
                STORE TMP
                CLEAR
                STORE INDEXTMP

CALC_INDEX_LOOP, LOAD TMP
                 SKIPCOND 400
                 JUMP CALC_INDEX_ADD_COL
                 LOAD INDEXTMP
                 ADD SIXTEEN
                 STORE INDEXTMP
                 LOAD TMP
                 SUBT ONE
                 STORE TMP
                 JUMP CALC_INDEX_LOOP

CALC_INDEX_ADD_COL, LOAD INDEXTMP
                    ADD COLUMNA
                    STORE INDEXTMP
                    JUMPI CALC_INDEX

/ ================================================================
/ VALIDATE_SHAPE_CELL
/ TMP = 1 si se puede jugar en esa celda
/ TMP = 0 si la celda está fuera de la forma
/ ================================================================
VALIDATE_SHAPE_CELL, HEX 0
                     LOAD INDEXTMP
                     ADD BASE_SHAPE
                     STORE DIR
                     LOADI DIR
                     STORE TMP
                     JUMPI VALIDATE_SHAPE_CELL

/ ================================================================
/ REVEAL_CELL
/ Si pisa mina -> GAME_STATE = 1
/ Si es celda segura y estaba oculta -> revelar y sumar contador
/ También imprime el número revelado o '*' si es mina
/ ================================================================
REVEAL_CELL,    HEX 0
                LOAD INDEXTMP
                ADD BASE_VIS
                STORE PTR_VIS
                LOADI PTR_VIS
                STORE TMP

                / No revelar si hay bandera
                LOAD TMP
                SUBT TWO
                SKIPCOND 400
                JUMP REVEAL_CHECK_ALREADY
                JUMPI REVEAL_CELL

REVEAL_CHECK_ALREADY, LOAD TMP
                      SUBT ONE
                      SKIPCOND 400
                      JUMP REVEAL_READ_REAL
                      JUMPI REVEAL_CELL

REVEAL_READ_REAL, LOAD INDEXTMP
                  ADD BASE_REAL
                  STORE PTR_REAL
                  LOADI PTR_REAL
                  STORE VALOR

                  LOAD VALOR
                  SKIPCOND 000
                  JUMP REVEAL_SAFE

                  / mina
                  LOAD ONE
                  STORE GAME_STATE
                  LOAD CH_STAR
                  OUTPUT
                  JUMPI REVEAL_CELL

REVEAL_SAFE,     LOAD ONE
                 STOREI PTR_VIS
                 LOAD REVEALED_COUNT
                 ADD ONE
                 STORE REVEALED_COUNT

                 / Imprimir valor 0..8 como número ASCII
                 LOAD VALOR
                 ADD 48
                 OUTPUT
                 JUMPI REVEAL_CELL

/ ================================================================
/ TOGGLE_FLAG
/ Si estaba oculta -> bandera
/ Si tenía bandera -> volver a oculta
/ Actualiza FLAG_COUNT e imprime score
/ ================================================================
TOGGLE_FLAG,    HEX 0
                LOAD INDEXTMP
                ADD BASE_VIS
                STORE PTR_VIS
                LOADI PTR_VIS
                STORE TMP

                / Si revelada, no hacer nada
                LOAD TMP
                SUBT ONE
                SKIPCOND 400
                JUMP TOGGLE_CHECK_FLAG
                JUMPI TOGGLE_FLAG

TOGGLE_CHECK_FLAG, LOAD TMP
                   SUBT TWO
                   SKIPCOND 400
                   JUMP PLACE_FLAG

                   / Quitar bandera
                   LOAD ZERO
                   STOREI PTR_VIS
                   LOAD FLAG_COUNT
                   SUBT ONE
                   STORE FLAG_COUNT
                   JNS PRINT_FLAGS
                   JUMPI TOGGLE_FLAG

PLACE_FLAG,     LOAD TWO
                STOREI PTR_VIS
                LOAD FLAG_COUNT
                ADD ONE
                STORE FLAG_COUNT
                JNS PRINT_FLAGS
                JUMPI TOGGLE_FLAG

/ ================================================================
/ CHECK_WIN
/ Gana si REVEALED_COUNT == SAFE_TARGET
/ ================================================================
CHECK_WIN,      HEX 0
                LOAD REVEALED_COUNT
                SUBT SAFE_TARGET
                SKIPCOND 400
                JUMP CHECK_WIN_EXIT
                LOAD TWO
                STORE GAME_STATE
CHECK_WIN_EXIT, JUMPI CHECK_WIN

/ ================================================================
/ CALC_SAFE_TARGET
/ Recorre SHAPE_PRESET y TABLERO_REAL
/ Cuenta cuántas casillas seguras activas existen.
/ ================================================================
CALC_SAFE_TARGET, HEX 0
                  CLEAR
                  STORE SAFE_TARGET
                  CLEAR
                  STORE COUNT

CALC_SAFE_LOOP,   LOAD COUNT
                  SUBT DEC256
                  SKIPCOND 000
                  JUMP CALC_SAFE_DONE

                  / leer shape
                  LOAD COUNT
                  ADD BASE_SHAPE
                  STORE PTR_SHAPE
                  LOADI PTR_SHAPE
                  STORE TMP

                  / si shape = 0, saltar
                  LOAD TMP
                  SKIPCOND 400
                  JUMP CALC_SAFE_NEXT

                  / leer tablero real
                  LOAD COUNT
                  ADD BASE_REAL
                  STORE PTR_REAL
                  LOADI PTR_REAL
                  STORE VALOR

                  / si valor = -1, es mina, no sumar
                  LOAD VALOR
                  SKIPCOND 000
                  JUMP CALC_SAFE_INC

                  JUMP CALC_SAFE_NEXT

CALC_SAFE_INC,    LOAD SAFE_TARGET
                  ADD ONE
                  STORE SAFE_TARGET

CALC_SAFE_NEXT,   LOAD COUNT
                  ADD ONE
                  STORE COUNT
                  JUMP CALC_SAFE_LOOP

CALC_SAFE_DONE,   JUMPI CALC_SAFE_TARGET

/ ================================================================
/ RENDER_BOARD
/ Recorre 256 celdas y pinta:
/ - fuera de forma -> COLOR_FORA
/ - oculta         -> COLOR_OCULTO
/ - bandera        -> COLOR_BANDERA
/ - revelada       -> color según 0..8
/ - si perdió y hay mina -> COLOR_MINA
/ ================================================================
RENDER_BOARD,    HEX 0
                 CLEAR
                 STORE COUNT

RENDER_LOOP,     LOAD COUNT
                 SUBT DEC256
                 SKIPCOND 000
                 JUMP RENDER_DONE

                 / Direcciones auxiliares
                 LOAD COUNT
                 ADD BASE_SHAPE
                 STORE PTR_SHAPE

                 LOAD COUNT
                 ADD BASE_VIS
                 STORE PTR_VIS

                 LOAD COUNT
                 ADD BASE_REAL
                 STORE PTR_REAL

                 LOAD COUNT
                 ADD BASE_DISP
                 STORE PTR_DISP

                 / shape?
                 LOADI PTR_SHAPE
                 STORE TMP
                 LOAD TMP
                 SKIPCOND 400
                 JUMP RENDER_FORA

                 / Si perdió, mostrar minas
                 LOAD GAME_STATE
                 SUBT ONE
                 SKIPCOND 400
                 JUMP RENDER_NORMAL
                 LOADI PTR_REAL
                 STORE VALOR
                 LOAD VALOR
                 SKIPCOND 000
                 JUMP RENDER_NORMAL
                 LOAD COLOR_MINA
                 STOREI PTR_DISP
                 JUMP RENDER_NEXT

RENDER_FORA,      LOAD COLOR_FORA
                  STOREI PTR_DISP
                  JUMP RENDER_NEXT

RENDER_NORMAL,    LOADI PTR_VIS
                  STORE TMP

                  / bandera
                  LOAD TMP
                  SUBT TWO
                  SKIPCOND 400
                  JUMP RENDER_CHECK_REVEALED
                  LOAD COLOR_BANDERA
                  STOREI PTR_DISP
                  JUMP RENDER_NEXT

RENDER_CHECK_REVEALED, LOAD TMP
                       SUBT ONE
                       SKIPCOND 400
                       JUMP RENDER_HIDDEN

                       / revelada -> decidir color según valor real
                       LOADI PTR_REAL
                       STORE VALOR

                       LOAD VALOR
                       SKIPCOND 400
                       JUMP RENDER_VAL_1
                       LOAD COLOR_0
                       STOREI PTR_DISP
                       JUMP RENDER_NEXT

RENDER_VAL_1,     LOAD VALOR
                  SUBT ONE
                  SKIPCOND 400
                  JUMP RENDER_VAL_2
                  LOAD COLOR_1
                  STOREI PTR_DISP
                  JUMP RENDER_NEXT

RENDER_VAL_2,     LOAD VALOR
                  SUBT TWO
                  SKIPCOND 400
                  JUMP RENDER_VAL_3
                  LOAD COLOR_2
                  STOREI PTR_DISP
                  JUMP RENDER_NEXT

RENDER_VAL_3,     LOAD VALOR
                  SUBT THREE
                  SKIPCOND 400
                  JUMP RENDER_VAL_4
                  LOAD COLOR_3
                  STOREI PTR_DISP
                  JUMP RENDER_NEXT

RENDER_VAL_4,     LOAD VALOR
                  SUBT FOUR
                  SKIPCOND 400
                  JUMP RENDER_VAL_5
                  LOAD COLOR_4
                  STOREI PTR_DISP
                  JUMP RENDER_NEXT

RENDER_VAL_5,     LOAD VALOR
                  SUBT FIVE
                  SKIPCOND 400
                  JUMP RENDER_VAL_6
                  LOAD COLOR_5
                  STOREI PTR_DISP
                  JUMP RENDER_NEXT

RENDER_VAL_6,     LOAD VALOR
                  SUBT SIX
                  SKIPCOND 400
                  JUMP RENDER_VAL_7
                  LOAD COLOR_6
                  STOREI PTR_DISP
                  JUMP RENDER_NEXT

RENDER_VAL_7,     LOAD VALOR
                  SUBT SEVEN
                  SKIPCOND 400
                  JUMP RENDER_VAL_8
                  LOAD COLOR_7
                  STOREI PTR_DISP
                  JUMP RENDER_NEXT

RENDER_VAL_8,     LOAD COLOR_8
                  STOREI PTR_DISP
                  JUMP RENDER_NEXT

RENDER_HIDDEN,    LOAD COLOR_OCULTO
                  STOREI PTR_DISP

RENDER_NEXT,      LOAD COUNT
                  ADD ONE
                  STORE COUNT
                  JUMP RENDER_LOOP

RENDER_DONE,      JUMPI RENDER_BOARD

/ ================================================================
/ SALIDAS DE TEXTO
/ ================================================================
PRINT_START,     HEX 0
                 LOAD CH_S
                 OUTPUT
                 LOAD CH_T
                 OUTPUT
                 LOAD CH_A
                 OUTPUT
                 LOAD CH_R
                 OUTPUT
                 LOAD CH_T
                 OUTPUT
                 LOAD CH_NEWLINE
                 OUTPUT
                 JUMPI PRINT_START

PRINT_WIN,       HEX 0
                 LOAD CH_W
                 OUTPUT
                 LOAD CH_I
                 OUTPUT
                 LOAD CH_N
                 OUTPUT
                 LOAD CH_NEWLINE
                 OUTPUT
                 JUMPI PRINT_WIN

PRINT_LOSE,      HEX 0
                 LOAD CH_L
                 OUTPUT
                 LOAD CH_O
                 OUTPUT
                 LOAD CH_S
                 OUTPUT
                 LOAD CH_E
                 OUTPUT
                 LOAD CH_NEWLINE
                 OUTPUT
                 JUMPI PRINT_LOSE

PRINT_FLAGS,     HEX 0
                 LOAD CH_F
                 OUTPUT
                 LOAD CH_L
                 OUTPUT
                 LOAD CH_A
                 OUTPUT
                 LOAD CH_G
                 OUTPUT
                 LOAD CH_S
                 OUTPUT
                 LOAD CH_COLON
                 OUTPUT
                 LOAD CH_SPACE
                 OUTPUT
                 LOAD FLAG_COUNT
                 ADD 48
                 OUTPUT
                 LOAD CH_NEWLINE
                 OUTPUT
                 JUMPI PRINT_FLAGS

END
"""
print("Plantilla cargada en memoria.")

## Generación del tablero, la máscara y la vista previa

In [ ]:

tablero = generar_tablero(
    filas=FILAS,
    columnas=COLUMNAS,
    minas=MINAS,
    semilla=SEMILLA,
    minas_fijas=MINAS_FIJAS,
)

mascara = generar_mascara(
    forma=FORMA,
    filas=FILAS,
    columnas=COLUMNAS,
    grosor=GROSOR,
    activas=ACTIVAS,
)

print("TABLERO REAL ('.' = fuera de forma):")
print(tablero_a_texto(tablero, mascara=mascara))

ruta_img = guardar_visualizacion(
    tablero,
    "outputs/vista_tablero_buscaminas.png",
    mascara=mascara,
    titulo=f"Buscaminas - forma: {FORMA}",
)

print(f"Imagen guardada en: {ruta_img}")


## Exportación de archivos MARIE

In [ ]:

def exportar_mas(template_str: str, tablero: Tablero, mascara: Mascara, ruta_salida: str | Path) -> Path:
    contenido = template_str.replace("__TABLERO_REAL__", bloque_dec_desde_matriz(tablero))
    contenido = contenido.replace("__SHAPE_PRESET__", bloque_dec_desde_matriz(mascara))
    ruta = Path(ruta_salida)
    ruta.write_text(contenido, encoding="utf-8")
    return ruta

ruta_mem_real = exportar_mem(tablero, "marie/generados/tablero_marie_generado.mem")
ruta_mem_shape = exportar_shape_mem(mascara, "marie/generados/shape_preset_generado.mem")
ruta_mas = exportar_mas(
    template_mas,
    tablero,
    mascara,
    "marie/generados/buscaminas_generado_mod.mas",
)

print("Archivos generados:")
print("-", ruta_mem_real)
print("-", ruta_mem_shape)
print("-", ruta_mas)


## Descarga rápida en Colab
Si estás en Colab, esta celda crea enlaces de descarga.

In [ ]:

try:
    from google.colab import files
    print("Entorno Colab detectado. Puedes descargar los archivos así:")
    print("files.download('marie/generados/buscaminas_generado_mod.mas')")
    print("files.download('marie/generados/tablero_marie_generado.mem')")
    print("files.download('marie/generados/shape_preset_generado.mem')")
except Exception:
    print("No estás en Colab. Los archivos ya quedaron guardados localmente.")
